**Import libraries**

In [1]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd

**Load your job/resume data**

In [2]:
df_jobs = pd.read_csv(r"C:\Users\bezis\Downloads\Smart-Resume-Job-Matching-System\data\cleaned_job_descriptions.csv")   # your jobs CSV
df_resumes = pd.read_csv(r"C:\Users\bezis\Downloads\Smart-Resume-Job-Matching-System\data\preprocessed_resumes.csv")        # your resumes CSV

**Initialize the model**

In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')  # fast and lightweight

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Encode your text**

In [4]:
job_embeddings = model.encode(df_jobs['clean_job_text'].tolist(), 
                              convert_to_tensor=True, batch_size=32)
resume_embeddings = model.encode(df_resumes['cleaned_text'].tolist(), 
                                 convert_to_tensor=True, batch_size=32)

**Compute similarity**

In [5]:
cos_scores = util.cos_sim(resume_embeddings, job_embeddings)

# Example: top 3 job matches for the first resume
top_results = cos_scores[0].topk(3)
print(top_results)

torch.return_types.topk(
values=tensor([0.6074, 0.5797, 0.5753]),
indices=tensor([566, 560, 855]))


In [6]:
from torch import nn

# job_embeddings: tensor of shape [num_jobs, embedding_dim]
# resume_embeddings: tensor of shape [num_resumes, embedding_dim]

cos = nn.CosineSimilarity(dim=1)  # computes similarity along dimension 1

# We'll compute similarity for each resume to all jobs
import torch

cosine_scores = []

for r_emb in resume_embeddings:
    # expand r_emb to match job_embeddings shape
    r_emb_expanded = r_emb.unsqueeze(0).expand(job_embeddings.size(0), -1)
    scores = cos(r_emb_expanded, job_embeddings)
    cosine_scores.append(scores)

# convert list of tensors to a single tensor: [num_resumes, num_jobs]
cosine_scores = torch.stack(cosine_scores)

**Store top matches in a DataFrame**

In [7]:
matches = []

for i, score_vector in enumerate(cosine_scores):
    top_k = score_vector.topk(3)  # top 3 matches
    for score, idx in zip(top_k[0], top_k[1]):
        matches.append({
            'resume_idx': i,
            'job_idx': idx.item(),
            'score': score.item(),
            'job_title': df_jobs.iloc[idx.item()]['Title']
        })

matches_df = pd.DataFrame(matches)
matches_df.head()

,resume_idx,job_idx,score,job_title
0,0,566,0.607404,Java Developer
1,0,560,0.579698,Java Developer
2,0,855,0.575315,Software Developer - Entry Level
3,1,566,0.707543,Java Developer
4,1,560,0.694333,Java Developer


In [8]:
import torch
import torch.nn.functional as F

batch_size = 100  # adjust to your RAM
all_scores = []

for i in range(0, len(resume_embeddings), batch_size):
    batch_resumes = resume_embeddings[i:i+batch_size]  # [batch_size, dim]
    # cosine similarity between batch and all jobs
    scores = F.cosine_similarity(
        batch_resumes.unsqueeze(1),  # [batch_size, 1, dim]
        job_embeddings.unsqueeze(0),  # [1, num_jobs, dim]
        dim=2
    )
    all_scores.append(scores)

# Combine all batches
cosine_scores = torch.cat(all_scores, dim=0)  # shape: [num_resumes, num_jobs]

In [9]:
top_k = 3  # Number of top matches to retrieve
matches = []

for i, score_vector in enumerate(cosine_scores):
    top_scores, top_indices = torch.topk(score_vector, top_k)
    for score, idx in zip(top_scores, top_indices):
        matches.append({
            'resume_idx': i,
            'job_idx': idx.item(),
            'score': score.item(),
            'job_title': df_jobs.iloc[idx.item()]['Title']
        })

matches_df = pd.DataFrame(matches)
matches_df.head()

,resume_idx,job_idx,score,job_title
0,0,566,0.607404,Java Developer
1,0,560,0.579698,Java Developer
2,0,855,0.575315,Software Developer - Entry Level
3,1,566,0.707543,Java Developer
4,1,560,0.694333,Java Developer


In [10]:
matches_df = pd.DataFrame({
    'resume_idx': [0, 0, 0, 1],
    'job_idx': [566, 560, 855, 566],
    'score': [0.607404, 0.579698, 0.575315, 0.707543],
    'job_title': ['Java Developer', 'Java Developer', 'Software Developer - Entry Level', 'Java Developer']
})


In [11]:
  # update if needed
matches_df['is_correct'] = [1, 1, 0, 1]

In [12]:
def precision_at_k(df, k=3):
    precision_list = []
    grouped = df.groupby('resume_idx')
    for _, group in grouped:
        correct = group['is_correct'][:k].sum()
        precision = correct / k
        precision_list.append(precision)
    return np.mean(precision_list)

In [13]:
def precision_at_k(df, k=3):
    precision_list = []
    grouped = df.groupby('resume_idx')
    for _, group in grouped:
        correct = group['is_correct'][:k].sum()
        precision = correct / k
        precision_list.append(precision)
    return np.mean(precision_list)

In [14]:
# Compute Mean Reciprocal Rank (MRR)
def mean_reciprocal_rank(df):
    mrr_list = []
    grouped = df.groupby('resume_idx')
    for _, group in grouped:
        ranks = group['is_correct'].to_list()
        rr = 0
        for i, val in enumerate(ranks):
            if val == 1:
                rr = 1 / (i + 1)
                break
        mrr_list.append(rr)
    return np.mean(mrr_list)

In [17]:
print("Precision@3:", precision_at_k(matches_df, k=3))
print("MRR:", mean_reciprocal_rank(matches_df))


Precision@3: 0.5
MRR: 1.0


**Threshold Filtering**

In [18]:
threshold = 0.6
matches_df['predicted_match'] = matches_df['score'] >= threshold

**Show only predicted good matches**

In [19]:
good_matches = matches_df[matches_df['predicted_match']]
print("\nPredicted Good Matches (score >= 0.6):")
print(good_matches)


Predicted Good Matches (score >= 0.6):
   resume_idx  job_idx     score       job_title  is_correct  predicted_match
0           0      566  0.607404  Java Developer           1             True
3           1      566  0.707543  Java Developer           1             True


In [20]:
import numpy  as np